### 1 - Preprocessing to save one dataset for  geometric + semantic info based on query Column  ( using Umap for reduction od dim)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = pd.read_parquet('ds/shopping_queries_dataset_examples.parquet')

In [3]:
#to focus only on one locale like us

data = data[data['product_locale'] == 'us']
data = data.sample(n=50000, random_state=42).reset_index(drop=True)

#data info

print(f"shape: {data.shape}")
print()
print(data.dtypes)

shape: (50000, 9)

example_id         int64
query             object
query_id           int64
product_id        object
product_locale    object
esci_label        object
small_version      int64
large_version      int64
split             object
dtype: object


In [4]:
data.head(2)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,2219601,workout weights,113894,B07HGX6G9K,us,E,1,1,train
1,1452229,nintendo gift cards,73849,B00K59HKIQ,us,I,0,1,train


In [5]:
#drop unessery columns since we don t use other dataset

# and also since we d ont want to use product dataset and our goal is only categorise based on query it self we
# don t need to use esci (exact , , , ....) labelling
data = data.drop(columns=['example_id' , 'product_id' , 'query_id' , 'split' , 'product_locale' ,'esci_label' , 'small_version' , 'large_version'])

In [6]:
data.info()

#we have now only one columns query

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   query   50000 non-null  object
dtypes: object(1)
memory usage: 390.8+ KB


In [7]:
#convert obj to string 
data['query'] = data['query'].astype('string')

In [8]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   query   50000 non-null  string
dtypes: string(1)
memory usage: 390.8 KB


In [9]:
data.isnull().sum()

query    0
dtype: int64

In [10]:
#geometrique Columns

# new columns from str columns
data['query_chars'] = data['query'].fillna('').str.len()
data['query_words'] = data['query'].fillna('').str.split().str.len()
#contain a number digit it could define or segment people the one that search for specifique like tv 50 inch 0 or  1
data['contains_digit'] = data['query'].str.contains(r'\d').astype(int)
#count digit
data['digit_count'] = data['query'].str.count(r'\d')

#average word length
data['avg_word_len'] = data['query_chars'] / data['query_words'].replace(0, 1)

#space count
data['space_count'] = data['query'].str.count(' ')

#special cracters count
data['spec_char_count'] = data['query'].str.count(r'[^a-zA-Z0-9\s]')

#upper case ratio no need everything low case
#data['upper_ratio'] = data['query'].str.findall(r'[A-Z]').str.len() / data['query_chars']


#since we are based on just query we tried to extract max info 

In [11]:
#semantic Column 

from sentence_transformers import SentenceTransformer

#it s a model that take a sentence and convert it to a vector of many dimmensions like dim1 take sentiment dim2 verb tense ....

#and this model contains 384 dimmension
# so simply it s a model takes text and convert to a numbre vector a meaning in math
#it s pretrained model with neural network


model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

embeddings = model.encode(
    data['query'].tolist(),
    batch_size=256,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

In [12]:
embeddings.shape 

(50000, 384)

In [13]:
embeddings[:10]

array([[-0.09854126,  0.0292511 , -0.0383825 , ..., -0.05338424,
         0.05466886, -0.00704982],
       [-0.10397693,  0.02758895, -0.01266059, ...,  0.01229009,
        -0.02926036,  0.0277749 ],
       [ 0.01959758,  0.0281667 ,  0.07780293, ..., -0.05311717,
        -0.02972028,  0.03857835],
       ...,
       [-0.02815098,  0.06356686, -0.01280949, ...,  0.06591015,
         0.06059738,  0.04284479],
       [-0.08132516,  0.02135177,  0.00382381, ...,  0.03616343,
         0.02651393, -0.00765904],
       [-0.10501935, -0.08706877, -0.01861359, ..., -0.04824757,
         0.02824977, -0.04335204]], shape=(10, 384), dtype=float32)

In [ ]:
from sklearn.preprocessing import normalize
from umap import UMAP

# reduce deminsions of embidings from 384 to 15

embeddings_norm = normalize(embeddings, norm='l2')

umap_model = UMAP(
    n_components=15,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)

emb_reduced = umap_model.fit_transform(embeddings_norm)

In [23]:
emb_reduced

array([[ 9.971348 ,  6.1912775,  5.3086033, ...,  5.2586074,  6.649301 ,
         3.7156413],
       [10.164223 ,  6.551484 ,  5.090791 , ...,  5.689204 ,  3.4772418,
         4.5649037],
       [10.004557 ,  6.5281744,  5.287583 , ...,  5.820741 ,  4.941647 ,
         2.841812 ],
       ...,
       [10.012732 ,  6.5623527,  5.1872473, ...,  5.8803873,  3.468994 ,
         4.4832253],
       [ 9.898094 ,  6.631697 ,  5.4932747, ...,  6.037125 ,  4.396642 ,
         4.118484 ],
       [10.043061 ,  6.4296126,  5.165868 , ...,  4.896134 ,  6.0706935,
         3.8607988]], shape=(50000, 15), dtype=float32)

In [24]:
#save it to dataframe

emb_df = pd.DataFrame(
    emb_reduced, 
    columns=[f'emb_{i}' for i in range(emb_reduced.shape[1])]
)

In [25]:
geo_features = [
    'query_chars', 'query_words', 'contains_digit',
    'digit_count', 'avg_word_len', 'space_count', 'spec_char_count'
]

In [26]:
from sklearn.preprocessing import StandardScaler

geo_cols = ['query_chars', 'query_words', 'contains_digit', 'digit_count' , 'avg_word_len' , 'space_count' , 'spec_char_count'] 

scaler = StandardScaler()
data[geo_cols] = scaler.fit_transform(data[geo_cols])

In [27]:
#save all to one dataframe

final_data = pd.concat(
    [data , emb_df],
    axis=1
)

In [29]:
final_data.head(8)

,query,query_chars,query_words,contains_digit,digit_count,avg_word_len,space_count,spec_char_count,emb_0,emb_1,...,emb_5,emb_6,emb_7,emb_8,emb_9,emb_10,emb_11,emb_12,emb_13,emb_14
0,workout weights,-0.651438,-0.917934,-0.419985,-0.365241,1.004975,-0.913646,-0.201665,9.971348,6.191278,...,4.896664,4.411188,4.665093,3.931290,3.995985,4.573812,2.660951,5.258607,6.649301,3.715641
1,nintendo gift cards,-0.250755,-0.346518,-0.419985,-0.365241,0.160833,-0.345260,-0.201665,10.164223,6.551484,...,4.146533,4.472170,4.676391,4.897981,3.552489,4.875464,0.991174,5.689204,3.477242,4.564904
2,urban skin rx,-0.851779,-0.346518,-0.419985,-0.365241,-1.286269,-0.345260,-0.201665,10.004557,6.528174,...,3.348886,4.228767,5.053605,4.204326,4.645471,3.953676,1.800899,5.820741,4.941647,2.841812
3,bird seed,-1.252462,-0.917934,-0.419985,-0.365241,-1.165677,-0.913646,-0.201665,9.942442,6.576708,...,5.001427,3.641950,5.439215,4.095617,4.588010,4.133751,1.432417,5.878413,5.180284,5.486079
4,+foot cream without alcohol,0.550610,0.224898,-0.419985,-0.365241,0.462312,0.223126,2.753563,10.062690,6.270202,...,5.384235,3.747347,6.095789,4.642343,5.536412,3.876423,3.241514,5.058719,6.594529,6.067585
5,brother tn730 high yield black toner,1.452146,1.367730,2.381036,3.337777,-0.080351,1.359897,-0.201665,10.043647,6.560628,...,3.050890,3.858025,5.223428,5.522382,5.241169,5.146523,2.270024,4.709783,4.631235,5.554499
6,60 lashes,-1.252462,-0.917934,2.381036,2.103437,-1.165677,-0.913646,-0.201665,10.060229,6.232531,...,4.976539,3.624479,6.647035,6.182129,6.455308,4.143440,3.635480,4.650783,6.156471,6.283861
7,cpap filters,-0.951950,-0.917934,-0.419985,-0.365241,-0.080351,-0.913646,-0.201665,10.135152,6.253580,...,5.874906,4.056500,6.612951,4.791937,6.211799,3.907670,3.114785,5.305024,6.119889,5.331352


In [30]:
final_data.to_csv("ds/cleaned_amazon_queries_dataset_en_50k_geo_semant_umap.csv")